In [ ]:
# Cài PyTorch tương thích với Tesla P100 (CUDA capability 6.0)
!pip install torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121
!pip install opencv-python

In [ ]:
!pip install -U ultralytics

In [ ]:
# ================================================================
# BƯỚC 1: TẠO ẢNH LOW-LIGHT TỔNG HỢP (theo phương pháp 3L-YOLO)
# Tăng cường dữ liệu thiếu sáng để model học nhận diện ban đêm
# ================================================================
import cv2
import numpy as np
import shutil
from pathlib import Path

def synthesize_low_light(image, brightness_range=(0.6, 0.8),
                         gamma_range=(2.0, 5.0),
                         noise_std_range=(0.1, 0.3)):
    """
    Tổng hợp ảnh thiếu sáng từ ảnh gốc theo phương pháp 3L-YOLO.
    4 bước: Đánh giá → Giảm sáng → Gamma → Thêm nhiễu.
    """
    img = image.astype(np.float32) / 255.0

    # Bước 1: Chỉ xử lý ảnh đủ sáng
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    if gray.mean() < 80:
        return image

    # Bước 2: Giảm sáng ngẫu nhiên xuống 60-80%
    brightness_factor = np.random.uniform(*brightness_range)
    img = img * brightness_factor

    # Bước 3: Gamma correction mô phỏng thiếu sáng
    gamma = np.random.uniform(*gamma_range)
    img = np.power(np.clip(img, 0, 1), gamma)

    # Bước 4: Thêm nhiễu Gaussian + Poisson
    noise_std = np.random.uniform(*noise_std_range)
    gaussian_noise = np.random.normal(0, noise_std, img.shape).astype(np.float32)
    img = img + gaussian_noise

    img_poisson = np.clip(img * 255, 0, 255).astype(np.uint8)
    img_poisson = np.random.poisson(img_poisson.astype(np.float32) / 10.0) * 10.0
    img = np.clip(img_poisson, 0, 255).astype(np.uint8)

    return img


def augment_dataset_low_light(src_img_dir, ratio=0.5):
    """
    Tạo ảnh low-light và lưu cùng thư mục gốc (cùng labels).
    ratio: tỷ lệ ảnh được tạo bản low-light.
    """
    src_path = Path(src_img_dir)
    label_dir = src_path.parent / "labels"

    images = list(src_path.glob("*.jpg")) + list(src_path.glob("*.png"))
    selected = np.random.choice(images, size=int(len(images) * ratio), replace=False)

    count = 0
    for img_path in selected:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        dark_img = synthesize_low_light(img)
        save_name = f"lowlight_{img_path.name}"
        cv2.imwrite(str(src_path / save_name), dark_img)

        # Copy label file tương ứng (giữ nguyên annotation)
        label_path = label_dir / img_path.with_suffix(".txt").name
        if label_path.exists():
            shutil.copy(label_path, label_dir / f"lowlight_{label_path.name}")
        count += 1

    print(f"✅ Đã tạo {count} ảnh low-light tổng hợp trong {src_path}")

In [ ]:
import os
import shutil as _shutil
from pathlib import Path
from ultralytics import YOLO

KAGGLE_INPUT = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")

# 1. Tìm file model .pt
MODEL_PATH = next(KAGGLE_INPUT.rglob("*.pt"), None)
if MODEL_PATH:
    print(f"✅ Đã tìm thấy model đính kèm: {MODEL_PATH}")
else:
    MODEL_PATH = "yolov8n.pt"
    print(f"⚠️ Không tìm thấy model đính kèm. Sẽ dùng pretrained: {MODEL_PATH}")

# 2. Tự động tìm thư mục dataset gốc (read-only)
DATASET_SRC = None
for p in KAGGLE_INPUT.rglob("train"):
    if p.is_dir() and (p / "images").exists():
        DATASET_SRC = p.parent
        break

if not DATASET_SRC:
    raise FileNotFoundError("Không tìm thấy thư mục dataset hợp lệ!")
print(f"✅ Dataset gốc (read-only): {DATASET_SRC}")

# 3. Copy dataset sang working dir (writable) để có thể augment
DATASET_PATH = WORKING_DIR / "dataset"
if not DATASET_PATH.exists():
    print("📋 Đang copy dataset sang /kaggle/working/dataset ...")
    _shutil.copytree(str(DATASET_SRC), str(DATASET_PATH))
    print(f"✅ Đã copy xong: {DATASET_PATH}")
else:
    print(f"✅ Dataset đã tồn tại: {DATASET_PATH}")

# 4. Tạo ảnh low-light tổng hợp cho tập train (50% ảnh)
augment_dataset_low_light(DATASET_PATH / "train" / "images", ratio=0.5)

In [ ]:
# 5. Tạo file YAML tự động (trỏ đến dataset đã copy trong working dir)
yaml_content = f"""
path: {DATASET_PATH}
train: train/images
val: val/images
nc: 7
names: ['pedestrian', 'bicycle', 'motorbike', 'bus', 'truck', 'container truck', 'car']
"""
YAML_FILE = WORKING_DIR / "dataset.yaml"
YAML_FILE.write_text(yaml_content.strip())

# 6. Load Model
model = YOLO(str(MODEL_PATH))

In [ ]:
# ================================================================
# TRAINING TỐI ƯU THEO 3L-YOLO CHO ĐIỀU KIỆN THIẾU SÁNG
# ================================================================
model.train(
    data=str(YAML_FILE),
    device=0,

    # === EPOCHS & PATIENCE (3L-YOLO: 200 epochs) ===
    epochs=200,
    patience=50,

    # === IMAGE SIZE ===
    imgsz=640,
    close_mosaic=15,

    # === BATCH & OPTIMIZER (theo 3L-YOLO) ===
    batch=24,
    optimizer="SGD",
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=5,
    cos_lr=True,

    # === AUGMENTATION TĂNG CƯỜNG CHO LOW-LIGHT ===
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.1,
    scale=0.6,
    translate=0.1,
    degrees=10.0,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    # === PERFORMANCE ===
    cache=True,
    workers=8,

    name="3l_yolo_traffic_lowlight"
)

In [ ]:
# ================================================================
# HIỂN THỊ KẾT QUẢ: Ma trận nhầm lẫn, PR curve, F1, ...
# ================================================================
from IPython.display import Image, display
from pathlib import Path

# Tìm thư mục kết quả (ưu tiên eval, fallback về training)
eval_dir = Path("/kaggle/working/runs/detect/3l_yolo_eval")
train_dir = Path("/kaggle/working/runs/detect/3l_yolo_traffic_lowlight")
save_dir = eval_dir if eval_dir.exists() else train_dir

plot_files = [
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "P_curve.png",
    "R_curve.png",
    "F1_curve.png",
    "results.png",
    "labels.jpg",
    "labels_correlogram.jpg",
    "val_batch0_pred.jpg",
]

for fname in plot_files:
    fpath = save_dir / fname
    if fpath.exists():
        print(f"\n{'='*60}")
        print(f"  {fname}")
        print(f"{'='*60}")
        display(Image(filename=str(fpath), width=800))
    else:
        print(f"[Không tìm thấy] {fname}")

In [ ]:
# ================================================================
# ĐÁNH GIÁ + TẠO LẠI BIỂU ĐỒ (confusion matrix, PR curve, F1...)
# ================================================================
best_weights = WORKING_DIR / "runs" / "detect" / "3l_yolo_traffic_lowlight" / "weights" / "best.pt"

if best_weights.exists():
    eval_model = YOLO(str(best_weights))
else:
    # Nếu không có best.pt, thử last.pt
    last_weights = WORKING_DIR / "runs" / "detect" / "3l_yolo_traffic_lowlight" / "weights" / "last.pt"
    if last_weights.exists():
        eval_model = YOLO(str(last_weights))
    else:
        eval_model = model

print(f"Model: {eval_model.ckpt_path}")

metrics = eval_model.val(
    data=str(YAML_FILE),
    conf=0.20,
    iou=0.5,
    device=0,
    plots=True,          # Tạo lại tất cả biểu đồ
    name="3l_yolo_eval"  # Lưu vào thư mục riêng
)
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")

In [ ]:
# ================================================================
# ĐÁNH GIÁ RIÊNG TRÊN TẬP VAL
# ================================================================
if best_weights.exists():
    eval_model = YOLO(str(best_weights))
else:
    eval_model = model

metrics = eval_model.val(
    data=str(YAML_FILE),
    conf=0.20,
    iou=0.5,
    device=0
)
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")